# ASL Translator: model testing on a free cloud GPU (Kaggle or Colab)

This notebook tests the translator's models on a free cloud GPU (T4/P100, 16 GB). It runs entirely inside the
notebook, which both platforms allow:

1. **Sentence models:** fine-tuned Qwen2.5-1.5B (deployed) vs stock **Qwen2.5-3B**, on the 115 held-out sentences.
2. **Word recognition:** accuracy on held-out signers, through the real server code, in Word and Sentence mode.
3. **Optional:** fine-tune Qwen2.5-3B the same way as the 1.5B, evaluate it, and download the adapter.

It does **not** host the live app. Colab's free tier restricts web UIs and proxies, and Kaggle has no clear rule.
See `docs/CLOUD.md`.

**Before running**
- *Kaggle:* Settings → Accelerator **GPU T4 x2** or **P100**; Settings → **Internet on** (needs a phone-verified
  account). Upload `asl_cloud_bundle_with_tests.tar.gz` as a **private** Kaggle dataset and add it to the notebook.
- *Colab:* Runtime → Change runtime type → **T4 GPU**. Put the bundle in Google Drive (cell 1 mounts it) or
  upload it when asked.
- Build the bundle on the lab PC: `python scripts/package_for_cloud.py --with-tests`, which writes
  `dist/asl_cloud_bundle_with_tests.tar.gz` (~380 MB).

In [ ]:
# 1. Find the bundle and unpack it
import glob, os, sys, subprocess, tarfile
IN_COLAB = "google.colab" in sys.modules
WORK = "/content" if IN_COLAB else "/kaggle/working"
if IN_COLAB:
    from google.colab import drive
    try: drive.mount("/content/drive")
    except Exception as e: print("Drive not mounted:", e)
cands = sorted(glob.glob("/kaggle/input/**/asl_cloud_bundle*.tar.gz", recursive=True)
               + glob.glob("/content/drive/MyDrive/**/asl_cloud_bundle*.tar.gz", recursive=True)
               + glob.glob("/content/asl_cloud_bundle*.tar.gz"))
if not cands and IN_COLAB:
    from google.colab import files
    up = files.upload(); cands = [os.path.abspath(k) for k in up]
assert cands, "bundle not found - add the Kaggle dataset / put it in Drive"
bundle = cands[-1]; print("bundle:", bundle)
with tarfile.open(bundle) as t: t.extractall(WORK)
os.chdir(f"{WORK}/ASL_Translator"); print("cwd:", os.getcwd())

In [ ]:
# 2. Install the server dependencies (keeps the platform's own CUDA PyTorch)
!pip install -q -r requirements-serve.txt
import nltk; nltk.download("wordnet", quiet=True)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch; print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| capability", torch.cuda.get_device_capability() if torch.cuda.is_available() else None)

## A. Sentence models: fine-tuned 1.5B vs stock 3B

Scores on 115 hand-written sentences that were never trained on.
- **coverage:** signed words kept. Higher is better.
- **added-content:** sentences where the model invented a word. Lower is better.
- **qform:** question vs statement correct. Higher is better.
- **chrF:** closeness to the reference. Higher is better.

On the lab PC: 1.5B + LoRA = 0.998 / 0.000 / 0.991 / 87.7 and stock 3B = 0.983 / 0.043 / 0.974 / 81.0.

In [ ]:
!python scripts/eval_composer.py --adapter checkpoints/composer/lora_1.5b --show 10 2>&1 | grep -v -i warn

In [ ]:
!python scripts/eval_composer.py --model Qwen/Qwen2.5-3B-Instruct --show 10 2>&1 | grep -v -i warn

## B. Word recognition on unseen signers (real server code)

- `ws_test.py N word`: one sign per trial in Word mode; reports how often exactly that word comes back.
- `ws_test.py N sentence`: three signs chained per trial; reports words found and exact sequences. The
  sentence model is loaded here too, so the 1.5B is used.
- `e2e_test.py`: the recognizer called directly on chained signs.

In [ ]:
!python scripts/ws_test.py 100 word 2>&1 | grep -a "word recall"
!python scripts/ws_test.py 60 sentence 2>&1 | grep -a "word recall"
!python scripts/e2e_test.py 100 2>&1 | grep -a "word recall"

## C. (Optional) Fine-tune Qwen2.5-3B for gloss → English

This is the same recipe as the deployed 1.5B: LoRA, lowest validation loss kept, early stopping. On T4/P100 it
runs in fp16 with loss scaling; on Ampere+ GPUs in bf16. It takes several hours on a T4, and Kaggle sessions last
up to ~12 h. Check that the session's time limit covers it before you start.

In [ ]:
!python scripts/make_sentence_data.py | head -1
!python scripts/train_composer.py --model Qwen/Qwen2.5-3B-Instruct --out checkpoints/composer/lora_3b --bs 4 --accum 8 --eval-every 100 --min-vram 9 2>&1 | grep -a -v -i warn | tail -15

In [ ]:
# Evaluate the fine-tuned 3B on the same 115 sentences (compare with section A)
!python scripts/eval_composer.py --model Qwen/Qwen2.5-3B-Instruct --adapter checkpoints/composer/lora_3b --show 10 2>&1 | grep -v -i warn

In [ ]:
# Package the 3B adapter to bring back to the lab PC (unpack into the project root there)
!tar -czf {WORK}/lora_3b.tar.gz checkpoints/composer/lora_3b && ls -la {WORK}/lora_3b.tar.gz
if IN_COLAB:
    from google.colab import files; files.download(f"{WORK}/lora_3b.tar.gz")
else:
    print("Kaggle: download lora_3b.tar.gz from the notebook's Output panel")

## On the lab PC afterwards

```bash
tar -xzf lora_3b.tar.gz                                        # in the project root
python -m src.serve.server --port 8000 --composer qwen-3b-lora   # needs ~8 GB free GPU memory
```
If the GPU does not have room, the server falls back to the fine-tuned 1.5B automatically.